# Genome PRD v1 — 03 XGBoost training

Train exactly the canonical 30-feature model and save it under `models/prd/`. Saved outputs predate the 30-feature extension. With the same data, environment, seed, and code the workflow is deterministic; byte-identical XGBoost serialization is not guaranteed across library or platform versions.

In [1]:
from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

here = Path.cwd().resolve()
ROOT = next(path for path in (here, *here.parents) if (path / 'src/training/prd_config.py').is_file())
sys.path.insert(0, str(ROOT))
from src.training import prd_config
from src.training.modeling import load_split

seed = int(prd_config.PRD_MODEL_PARAMS['random_state'])
np.random.seed(seed)
ratings = pd.read_parquet(prd_config.RATINGS_SOURCE_PATH, columns=['rating'])['rating'].to_numpy(dtype=np.float32, copy=False)

## Load canonical train and validation matrices

In [2]:
X_train, y_train, train_time = load_split(prd_config.FEATURE_ARTIFACT_PATH, ratings, 'train', prd_config.PRD_FEATURES)
X_validation, y_validation, validation_time = load_split(prd_config.FEATURE_ARTIFACT_PATH, ratings, 'validation', prd_config.PRD_FEATURES)
display(pd.DataFrame([
    {'split': 'train', 'rows': len(y_train), 'prevalence': y_train.mean(), 'min_time': train_time.min(), 'max_time': train_time.max()},
    {'split': 'validation', 'rows': len(y_validation), 'prevalence': y_validation.mean(), 'min_time': validation_time.min(), 'max_time': validation_time.max()},
]))
assert X_train.shape[1] == X_validation.shape[1] == len(prd_config.PRD_FEATURES) == 30

,split,rows,prevalence,min_time,max_time
0,train,17822773,0.498194,1995-01-09 11:46:44,2011-12-31 23:59:55
1,validation,1330716,0.519539,2012-01-01 00:00:40,2013-12-31 23:59:59


## Fit with the canonical parameters and validation early stopping

In [3]:
model = XGBClassifier(**prd_config.PRD_MODEL_PARAMS)
started = time.perf_counter()
model.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_validation, y_validation)], verbose=25)
training_runtime_seconds = time.perf_counter() - started
training_summary = pd.Series({'runtime_seconds': training_runtime_seconds, 'best_iteration_zero_based': int(model.best_iteration), 'effective_tree_count': int(model.get_booster().num_boosted_rounds())})
display(training_summary.to_frame('value'))
#validation_0 → train (1995–2011)
#validation_1 → validation temporal (2012–2013)

[0]	validation_0-logloss:0.68040	validation_0-auc:0.79444	validation_0-aucpr:0.77819	validation_1-logloss:0.68064	validation_1-auc:0.79023	validation_1-aucpr:0.78894
[25]	validation_0-logloss:0.55370	validation_0-auc:0.80885	validation_0-aucpr:0.79493	validation_1-logloss:0.55727	validation_1-auc:0.80486	validation_1-aucpr:0.80591
[50]	validation_0-logloss:0.53104	validation_0-auc:0.81242	validation_0-aucpr:0.79917	validation_1-logloss:0.53433	validation_1-auc:0.80928	validation_1-aucpr:0.81087
[75]	validation_0-logloss:0.52477	validation_0-auc:0.81440	validation_0-aucpr:0.80161	validation_1-logloss:0.52777	validation_1-auc:0.81165	validation_1-aucpr:0.81376
[100]	validation_0-logloss:0.52215	validation_0-auc:0.81571	validation_0-aucpr:0.80320	validation_1-logloss:0.52479	validation_1-auc:0.81335	validation_1-aucpr:0.81571
[125]	validation_0-logloss:0.52061	validation_0-auc:0.81667	validation_0-aucpr:0.80434	validation_1-logloss:0.52298	validation_1-auc:0.81457	validation_1-aucpr:0.816

,value
runtime_seconds,2788.154284
best_iteration_zero_based,599.000000
effective_tree_count,600.000000


## Save the canonical PRD model artifact

In [5]:
prd_config.PRD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_model(prd_config.PRD_ARTIFACT_PATH)
print('Saved:', prd_config.PRD_ARTIFACT_PATH.relative_to(ROOT))

Saved: models/prd/xgboost_genome_prd_v1.model.json
